In [3]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine

### Import txt file and manage data

In [ ]:
# import txt file
column_names = ['name', 'var_1', 'official_name_stop', 'var_2', 'latitude', 
                'longitude', 'var_3', 'var_4']
df = pd.read_csv('../data/CTA_STOP_XFERS.txt', header = None, names = column_names)

# remove unnecessary fields and add necessary fields
df = df[['name', 'latitude', 'longitude']]
df['type'] = 'CTA'

# keep only data within Hyde Park
HP_NORTH_BOUND = 41.809647
HP_SOUTH_BOUND = 41.780482
HP_WEST_BOUND = -87.615877
HP_EAST_BOUND = -87.579056
df = df[
    (df['latitude'] >= HP_SOUTH_BOUND) & (df['latitude'] <= HP_NORTH_BOUND) &
    (df['longitude'] >= HP_WEST_BOUND) & (df['longitude'] <= HP_EAST_BOUND)
]
df

,name,latitude,longitude,type
356,171,41.794720,-87.591858,CTA
357,171,41.795036,-87.592962,CTA
358,171,41.795109,-87.587858,CTA
359,171,41.795037,-87.596250,CTA
360,171,41.787871,-87.594547,CTA
...,...,...,...,...
13667,X4,41.802372,-87.606422,CTA
13668,X4,41.795203,-87.606263,CTA
13669,X4,41.789442,-87.606105,CTA
13670,X4,41.784150,-87.606057,CTA


### Ingest data into SQL

In [5]:
# Load .env
load_dotenv()
db_url = os.getenv("DATABASE_URL")

# Create SQLAlchemy engine
engine = create_engine(db_url)

# Create table and ingest the data to thetable
df.to_sql("apt_app_cta_stops", engine, if_exists='replace', index=False)

416

### Check SQL data

In [7]:
# Count how many rows has the SQL table
db_count = pd.read_sql("SELECT COUNT(*) AS count FROM apt_app_cta_stops", engine).iloc[0, 0]
print("Total rows:", db_count)

# head
df_preview = pd.read_sql("SELECT * FROM apt_app_cta_stops LIMIT 5", engine)
print(df_preview)

Total rows: 416
  name   latitude  longitude type
0  171  41.794720 -87.591858  CTA
1  171  41.795036 -87.592962  CTA
2  171  41.795109 -87.587858  CTA
3  171  41.795037 -87.596250  CTA
4  171  41.787871 -87.594547  CTA
